In [14]:
import pandas as pd
df=pd.read_csv('combined_incomplete2.csv')
#change this to the full thing later, rn just to test

reliability_summary=df.groupby('edge_list').agg({
    'atr':['min','max','mean'],
    'num_edges': 'first'
}).reset_index()

reliability_summary.columns=['edge_list','atr_min','atr_max','atr_mean','num_edges']

In [15]:
top_graphs=reliability_summary.sort_values(by='atr_mean',ascending=False).head(10)

bottom_graphs=reliability_summary.sort_values(by='atr_mean').head(10)

In [16]:
import ast
import networkx as nx

def parse_graph(edge_str):
    edges=ast.literal_eval(edge_str)
    G=nx.Graph()
    G.add_edges_from(edges)
    return G


In [17]:
conda install scipy

Retrieving notices: done
Channels:
 - defaults
Platform: osx-arm64
Solving environment: done

## Package Plan ##

  environment location: /opt/anaconda3/envs/graph_nodes

  added / updated specs:
    - scipy


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2025.7.15  |       hca03da5_0         127 KB
    certifi-2025.8.3           |  py313hca03da5_0         161 KB
    ------------------------------------------------------------
                                           Total:         288 KB

The following packages will be UPDATED:

  ca-certificates                      2025.2.25-hca03da5_0 --> 2025.7.15-hca03da5_0 
  certifi                         2025.7.14-py313hca03da5_0 --> 2025.8.3-py313hca03da5_0 



certifi-2025.8.3     | 161 KB    |                                       |   0% 
ca-certificates-2025 | 127 KB    |                                       |   0% 
cer

In [18]:
import numpy as np
import scipy.linalg

def compute_lambda_t(G):
    lambda_val=nx.edge_connectivity(G)
    L=nx.laplacian_matrix(G).toarray() #find laplacian matrix
    L_minor=np.delete(np.delete(L, 0, axis=0), 0, axis=1)
    t_val=round(np.linalg.det(L_minor))
    #determinant of minor is the number of spanning trees
    return lambda_val, int(t_val)

reliability_summary['G']=reliability_summary['edge_list'].apply(parse_graph)
reliability_summary[['lambda','tree_number']]=reliability_summary['G'].apply(
    lambda G: pd.Series(compute_lambda_t(G))
)


In [19]:
reliability_summary['lambda_optimal']=False
reliability_summary['t_optimal']=False

for e_val, group in reliability_summary.groupby('num_edges'):
    max_lambda = group['lambda'].max()
    max_t=group['tree_number'].max()

    reliability_summary.loc[group.index, 'lambda_optimal']=(group['lambda']==max_lambda)
    reliability_summary.loc[group.index, 't_optimal']=(group['tree_number']==max_t)

In [20]:
reliability_summary[['edge_list','atr_mean','lambda','tree_number','lambda_optimal','t_optimal']].sort_values(by='atr_mean', ascending=False).head(10)

,edge_list,atr_mean,lambda,tree_number,lambda_optimal,t_optimal
98,"[(0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (1, 3...",0.643634,5,102400,True,True
99,"[(0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (1, 3...",0.634151,5,76800,True,True
14,"[(0, 2), (0, 4), (0, 5), (0, 6), (0, 7), (1, 3...",0.630988,5,73728,True,False
121,"[(0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (1, 3...",0.625861,4,71680,False,False
100,"[(0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (1, 3...",0.625059,5,57600,True,True
102,"[(0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (1, 3...",0.622256,5,56000,True,False
15,"[(0, 2), (0, 4), (0, 5), (0, 6), (0, 7), (1, 3...",0.621984,5,55296,True,False
122,"[(0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (1, 3...",0.617048,4,53760,False,False
126,"[(0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (1, 3...",0.614431,4,52160,False,False
21,"[(0, 2), (0, 4), (0, 5), (0, 6), (0, 7), (1, 3...",0.613963,4,51456,False,False


In [21]:
reliability_summary.to_csv('partial_data_labels1.csv',index=False)

In [ ]:
#lambda_optimal = True if they have the maximum λ among all graphs with the same number of nodes and edges.
#t_optimal = True if they have the maximum tree number in that class.

#lambda = edge connectivity; t=number of spanning trees